# 02 — Dengue Data Validation

## Dengue Disease Prediction for Sri Lanka

#### Purpose

This notebook performs detailed validation of the raw weekly dengue dataset before geographic standardization and monthly aggregation.

The validation checks include:

- dataset structure
- temporal coverage
- epidemiological week coverage
- date validity
- dengue case values
- duplicate observations
- reporting area coverage
- district week completeness

The raw dengue dataset is not modified in this notebook.

In [101]:
import pandas as pd
from pathlib import Path

In [102]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DENGUE_DIR = PROJECT_ROOT / "data" / "raw" / "dengue"
print("Project root:", PROJECT_ROOT)
print("Dengue folder:", DENGUE_DIR)

Project root: d:\Dengue_Project
Dengue folder: d:\Dengue_Project\data\raw\dengue


In [103]:
dengue_file = list(DENGUE_DIR.glob("*.csv"))[0]
print("Dengue file:", dengue_file.name)

Dengue file: srilanka_dengue_weekly_2010_2024.csv


In [104]:
dengue = pd.read_csv(dengue_file)
print("Shape:", dengue.shape)

Shape: (20306, 6)


In [105]:
expected_columns = [
    "year",
    "week",
    "start.date",
    "end.date",
    "district",
    "cases"
]
print("Actual columns:")
print(dengue.columns.tolist())
print("\nExpected columns:")
print(expected_columns)

Actual columns:
['year', 'week', 'start.date', 'end.date', 'district', 'cases']

Expected columns:
['year', 'week', 'start.date', 'end.date', 'district', 'cases']


## 1. Structural Validation

The raw dengue dataset is checked for its expected columns, data types, missing values and duplicate rows.

In [106]:
print(dengue.isnull().sum())

year          0
week          0
start.date    0
end.date      0
district      0
cases         0
dtype: int64


In [107]:
print("Total missing values:", dengue.isnull().sum().sum())

Total missing values: 0


In [108]:
print("Duplicate rows:", dengue.duplicated().sum())

Duplicate rows: 0


## 2. Temporal Validation

This section validates the year and epidemiological week coverage of the dengue dataset.

In [109]:
print("Minimum year:", dengue["year"].min())
print("Maximum year:", dengue["year"].max())

Minimum year: 2010
Maximum year: 2024


In [110]:
print(sorted(dengue["year"].unique()))

[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


Check record counts per year

In [111]:
records_per_year = dengue["year"].value_counts().sort_index()

print(records_per_year)

year
2010    1352
2011    1352
2012    1352
2013    1352
2014    1352
2015    1352
2016    1378
2017    1352
2018    1352
2019    1352
2020    1352
2021    1378
2022    1352
2023    1326
2024    1352
Name: count, dtype: int64


Validate week numbers

In [112]:
print("Minimum week:", dengue["week"].min())
print("Maximum week:", dengue["week"].max())

Minimum week: 1
Maximum week: 53


Count weeks in each year

In [113]:
weeks_per_year = dengue.groupby("year")["week"].nunique()
print(weeks_per_year)

year
2010    52
2011    52
2012    52
2013    52
2014    52
2015    52
2016    53
2017    52
2018    52
2019    52
2020    52
2021    53
2022    52
2023    51
2024    52
Name: week, dtype: int64


Automatically identify unusual years

In [114]:
unusual_years = weeks_per_year[weeks_per_year != 52]
print(unusual_years)

year
2016    53
2021    53
2023    51
Name: week, dtype: int64


Validate dates

In [115]:
start_dates = pd.to_datetime(
    dengue["start.date"],
    errors="coerce"
)
end_dates = pd.to_datetime(
    dengue["end.date"],
    errors="coerce"
)

In [116]:
print("Invalid start dates:", start_dates.isnull().sum())
print("Invalid end dates:", end_dates.isnull().sum())

Invalid start dates: 0
Invalid end dates: 0


Check date ordering

In [117]:
invalid_date_order = (end_dates < start_dates).sum()
print("Rows where end date is before start date:", invalid_date_order)

Rows where end date is before start date: 0


Check the length of each reporting period

In [118]:
period_length = (end_dates - start_dates).dt.days
print(period_length.value_counts().sort_index())

6    20306
Name: count, dtype: int64


## 3. Dengue Case Validation

This section checks whether the dengue case counts contain missing, negative, or otherwise suspicious values.

In [119]:
print(dengue["cases"].describe())

count    20306.000000
mean        35.198808
std         86.947966
min          0.000000
25%          3.000000
50%         11.000000
75%         33.000000
max       2631.000000
Name: cases, dtype: float64


In [120]:
negative_cases = (dengue["cases"] < 0).sum()
print("Negative case values:", negative_cases)

Negative case values: 0


In [121]:
zero_cases = (dengue["cases"] == 0).sum()
print("Zero-case observations:", zero_cases)
zero_percentage = (zero_cases / len(dengue)) * 100
print("Zero-case percentage:", round(zero_percentage, 2), "%")

Zero-case observations: 2208
Zero-case percentage: 10.87 %


In [122]:
print("Cases data type:", dengue["cases"].dtype)

Cases data type: int64


## 4. Reporting-Area Validation

This section examines the geographic reporting areas represented in the dengue dataset.

In [123]:
print("Number of reporting areas:", dengue["district"].nunique())

Number of reporting areas: 26


In [124]:
for area in sorted(dengue["district"].unique()):
    print(area)

Ampara
Anuradhapura
Badulla
Batticaloa
Colombo
Galle
Gampaha
Hambanthota
Jaffna
Kalmune
Kalutara
Kandy
Kegalle
Kilinochchi
Kurunegala
Mannar
Matale
Matara
Monaragala
Mullaitivu
NuwaraEliya
Polonnaruwa
Puttalam
Ratnapura
Trincomalee
Vavuniya


Check duplicate district-week keys

In [125]:
duplicate_keys = dengue.duplicated(
    subset=["year", "week", "district"]
).sum()
print("Duplicate year-week-district combinations:", duplicate_keys)

Duplicate year-week-district combinations: 0


In [126]:
areas_per_week = dengue.groupby(
    ["year", "week"]
)["district"].nunique()
print(areas_per_week.value_counts().sort_index())

district
26    781
Name: count, dtype: int64


Specifically validate 2023 Week 52

In [127]:
week_52_2023 = dengue[
    (dengue["year"] == 2023) &
    (dengue["week"] == 52)
]
print("Rows for 2023 Week 52:", len(week_52_2023))

Rows for 2023 Week 52: 0


In [128]:
week_51_2023 = dengue[
    (dengue["year"] == 2023) &
    (dengue["week"] == 51)
]
print("Rows for 2023 Week 51:", len(week_51_2023))

Rows for 2023 Week 51: 26


Check cases by year

In [129]:
yearly_cases = dengue.groupby("year")["cases"].sum()
print(yearly_cases)

year
2010     25006
2011     21647
2012     23527
2013     22387
2014     35314
2015     22313
2016     35192
2017    174687
2018     51321
2019     94780
2020     32222
2021     20213
2022     31674
2023     70918
2024     53546
Name: cases, dtype: int64


### Observation: Reporting Period

All 20,306 dengue observations have a 6 day difference between the recorded start and end dates, corresponding to a 7 day inclusive weekly reporting period. No invalid date ordering was detected.

### Observation: Dengue Case Values

No negative dengue case counts were detected. A total of 2,208 observations (10.87%) contain zero reported cases. Zero values are retained as valid observations rather than treated as missing values.

The case distribution contains substantially larger values during some reporting periods, with a maximum weekly reporting area count of 2,631 cases. High case counts are not automatically treated as errors because they may represent genuine dengue outbreak periods. Their distribution will be examined further during exploratory data analysis.

## 5. Date and Week Consistency Validation

This section checks whether the temporal identifiers (`year` and `week`) are internally consistent with the recorded weekly date ranges.

In [130]:
date_consistency = dengue.groupby(
    ["year", "week"]
).agg(
    unique_start_dates=("start.date", "nunique"),
    unique_end_dates=("end.date", "nunique")
)
date_consistency.head()

unique_start_dates  unique_end_dates
year week                                      
2010 1                      1                 1
     2                      1                 1
     3                      1                 1
     4                      1                 1
     5                      1                 1

In [131]:
inconsistent_week_dates = date_consistency[
    (date_consistency["unique_start_dates"] != 1) |
    (date_consistency["unique_end_dates"] != 1)
]
print("Year-week periods with inconsistent dates:", len(inconsistent_week_dates))

Year-week periods with inconsistent dates: 0


### Observation

Each represented year week combination uses a single consistent start date and end date across all reporting areas.

Create one row per epidemiological week

In [132]:
weekly_calendar = dengue[
    ["year", "week", "start.date", "end.date"]
].drop_duplicates().copy()
weekly_calendar["start_date"] = pd.to_datetime(
    weekly_calendar["start.date"],
    errors="coerce"
)
weekly_calendar["end_date"] = pd.to_datetime(
    weekly_calendar["end.date"],
    errors="coerce"
)
weekly_calendar = weekly_calendar.sort_values(
    ["year", "week"]
).reset_index(drop=True)
weekly_calendar.head(10)

,year,week,start.date,end.date,start_date,end_date
0,2010,1,1/2/2010,1/8/2010,2010-01-02,2010-01-08
1,2010,2,1/9/2010,1/15/2010,2010-01-09,2010-01-15
2,2010,3,1/16/2010,1/22/2010,2010-01-16,2010-01-22
3,2010,4,1/23/2010,1/29/2010,2010-01-23,2010-01-29
4,2010,5,1/30/2010,2/5/2010,2010-01-30,2010-02-05
5,2010,6,2/6/2010,2/12/2010,2010-02-06,2010-02-12
6,2010,7,2/13/2010,2/19/2010,2010-02-13,2010-02-19
7,2010,8,2/20/2010,2/26/2010,2010-02-20,2010-02-26
8,2010,9,2/27/2010,3/5/2010,2010-02-27,2010-03-05
9,2010,10,3/6/2010,3/12/2010,2010-03-06,2010-03-12


In [133]:
print("Unique reporting weeks:", len(weekly_calendar))

Unique reporting weeks: 781


Inspect year boundaries

In [134]:
boundary_weeks = weekly_calendar[
    weekly_calendar["week"].isin([1, 52, 53])
]
boundary_weeks[
    ["year", "week", "start_date", "end_date"]
].tail(30)

,year,week,start_date,end_date
51,2010,52,2010-12-25,2010-12-31
52,2011,1,2011-01-01,2011-01-07
103,2011,52,2011-12-24,2011-12-30
104,2012,1,2011-12-31,2012-01-06
155,2012,52,2012-12-22,2012-12-28
156,2013,1,2012-12-29,2013-01-04
207,2013,52,2013-12-21,2013-12-27
208,2014,1,2013-12-28,2014-01-03
259,2014,52,2014-12-20,2014-12-26
260,2015,1,2014-12-27,2015-01-02


Check whether the reported year agrees with the dates

In [135]:
weekly_calendar["start_year"] = weekly_calendar["start_date"].dt.year
weekly_calendar["end_year"] = weekly_calendar["end_date"].dt.year

start_year_mismatch = (
    weekly_calendar["year"] != weekly_calendar["start_year"]
).sum()

end_year_mismatch = (
    weekly_calendar["year"] != weekly_calendar["end_year"]
).sum()

print("Reported year different from start-date year:", start_year_mismatch)
print("Reported year different from end-date year:", end_year_mismatch)

Reported year different from start-date year: 13
Reported year different from end-date year: 1


Check chronological ordering

In [136]:
calendar_by_date = weekly_calendar.sort_values(
    "start_date"
).reset_index(drop=True)

calendar_by_date[
    ["year", "week", "start_date", "end_date"]
].head()

,year,week,start_date,end_date
0,2010,1,2010-01-02,2010-01-08
1,2010,2,2010-01-09,2010-01-15
2,2010,3,2010-01-16,2010-01-22
3,2010,4,2010-01-23,2010-01-29
4,2010,5,2010-01-30,2010-02-05


In [137]:
print("Earliest start date:", calendar_by_date["start_date"].min())
print("Latest end date:", calendar_by_date["end_date"].max())

Earliest start date: 2010-01-02 00:00:00
Latest end date: 2024-12-20 00:00:00


In [138]:
calendar_by_date["days_from_previous_week"] = (
    calendar_by_date["start_date"].diff().dt.days
)
calendar_by_date["days_from_previous_week"].value_counts().sort_index()

days_from_previous_week
7.0    780
Name: count, dtype: int64

In [139]:
large_gaps = calendar_by_date[
    calendar_by_date["days_from_previous_week"] > 7
]
large_gaps[
    [
        "year",
        "week",
        "start_date",
        "end_date",
        "days_from_previous_week"
    ]
]

,year,week,start_date,end_date,days_from_previous_week


Check for overlapping weekly periods

In [140]:
calendar_by_date["previous_end_date"] = (
    calendar_by_date["end_date"].shift(1)
)

overlapping_periods = calendar_by_date[
    calendar_by_date["start_date"] <=
    calendar_by_date["previous_end_date"]
]

print("Overlapping reporting periods:", len(overlapping_periods))

Overlapping reporting periods: 0


Identify missing week numbers within each year


In [141]:
missing_weeks = {}
for year in sorted(dengue["year"].unique()):
    observed_weeks = set(
        dengue.loc[dengue["year"] == year, "week"].unique()
    )
    max_week = max(observed_weeks)
    expected_weeks = set(range(1, max_week + 1))
    missing = sorted(expected_weeks - observed_weeks)
    if missing:
        missing_weeks[year] = missing
print("Missing week numbers:")
print(missing_weeks)

Missing week numbers:
{}


In [142]:
print("Number of dengue observations:", len(dengue))
print("Total dengue cases:", dengue["cases"].sum())

Number of dengue observations: 20306
Total dengue cases: 714747


In [143]:
count_2023_week51 = len(
    dengue[
        (dengue["year"] == 2023) &
        (dengue["week"] == 51)
    ]
)
count_2023_week52 = len(
    dengue[
        (dengue["year"] == 2023) &
        (dengue["week"] == 52)
    ]
)
print("2023 Week 51 records:", count_2023_week51)
print("2023 Week 52 records:", count_2023_week52)

2023 Week 51 records: 26
2023 Week 52 records: 0


In [144]:
validation_summary = pd.DataFrame({
    "Validation Check": [
        "Total observations",
        "Total columns",
        "Standard missing values",
        "Exact duplicate rows",
        "Duplicate year-week-area keys",
        "Negative case values",
        "Zero-case observations",
        "Invalid start dates",
        "Invalid end dates",
        "End date before start date",
        "Reporting areas",
        "Unique represented weeks",
        "Inconsistent dates within year-week",
        "2023 Week 52 records"
    ],
    "Result": [
        len(dengue),
        len(dengue.columns),
        dengue.isnull().sum().sum(),
        dengue.duplicated().sum(),
        duplicate_keys,
        negative_cases,
        zero_cases,
        start_dates.isnull().sum(),
        end_dates.isnull().sum(),
        invalid_date_order,
        dengue["district"].nunique(),
        len(weekly_calendar),
        len(inconsistent_week_dates),
        count_2023_week52
    ]
})

validation_summary

,Validation Check,Result
0,Total observations,20306
1,Total columns,6
2,Standard missing values,0
3,Exact duplicate rows,0
4,Duplicate year-week-area keys,0
5,Negative case values,0
6,Zero-case observations,2208
7,Invalid start dates,0
8,Invalid end dates,0
9,End date before start date,0


## Key Validation Findings

1. The raw dengue dataset contains 20,306 observations and 6 variables covering the period 2010–2024.

2. No standard null (`NaN`) values or exact duplicate rows were detected.

3. No duplicate `year week reporting area` combinations were detected, indicating that each represented reporting area/week combination occurs only once.

4. No negative dengue case counts were detected. A total of 2,208 observations (10.87%) contain zero reported cases. These zero values are retained as valid observations rather than interpreted as missing data.

5. All recorded weekly periods have valid start and end dates, with no end date occurring before its corresponding start date.

6. All 20,306 observations have a 6 day difference between start and end dates, corresponding to a 7 day inclusive reporting period.

7. The dataset contains 26 reporting areas. Every epidemiological week represented in the dataset contains records for all 26 reporting areas.

8. The source reporting labels contain 53 weeks in 2016 and 2021 and 51 weeks under the 2023 year label. However, chronological date validation shows that consecutive reporting periods remain exactly 7 days apart throughout the dataset.

9. No record is labelled as Week 52 of 2023 in the dengue dataset. However, chronological validation identified no missing 7-day period at this boundary. The period following the final 2023-labelled week is represented as Week 1 of 2024. Therefore, the absence of a `2023 Week 52` label should not be interpreted as a missing dengue observation.

10. Geographic names are retained exactly as provided by the source in this notebook. Geographic standardization will be performed separately.

11. High dengue case counts are not automatically treated as errors because they may represent genuine outbreak periods. Their distributions will be investigated during exploratory data analysis.

## Conclusion

The raw dengue dataset demonstrates strong internal structural consistency. No standard null values, exact duplicate rows, duplicate year week reporting-area keys, negative case counts, invalid dates, partially reported weeks, overlapping weekly periods or chronological gaps greater than seven days were identified.

Although no dengue records are labelled as Week 52 of 2023, chronological validation shows that the weekly sequence remains continuous, with consecutive reporting periods separated by exactly seven days. The subsequent period is labelled as Week 1 of 2024. Therefore, the absence of the `2023 Week 52` label is not treated as evidence of a missing dengue observation.

The validation also shows that source reporting year labels do not always correspond directly to the calendar year of the weekly start date, particularly around year boundaries. This reporting convention must be considered when aligning dengue observations with weather data. Temporal integration should therefore be based on validated date periods rather than assuming that `year` and `week` labels are directly equivalent across datasets.

Geographic naming inconsistencies identified during the data inventory stage are intentionally left unchanged because geographic harmonization will be performed separately.

No raw dengue records were modified during this validation process.